# オープンロジ出荷依頼CSVパイプライン 検証ノートブック

`build_plans_csv.py` → `build_opl_csv.py` の一連の処理を、`sample/`・`mock/fixtures/` のダミーデータで動かし、SKILL.mdに記載された業務ルール（緩衝材の同梱揃え等）が正しく再現されるかを検証する。

依存ライブラリは標準ライブラリのみ（pandas等は使わない）。

In [ ]:
import csv, io, json, os, subprocess, sys, tempfile
from IPython.display import display, HTML

HERE = os.getcwd()
PLANS_SCRIPT = os.path.join(HERE, "build_plans_csv.py")
OPL_SCRIPT = os.path.join(HERE, "build_opl_csv.py")
SAMPLE = os.path.join(HERE, "sample")
FIXTURES = os.path.join(HERE, "mock", "fixtures")

def run(args):
    r = subprocess.run([sys.executable] + args, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stdout + r.stderr)
    return r.stdout

def read_csv_rows(path, enc="utf-8-sig", skip=0):
    with open(path, encoding=enc, newline="") as f:
        rows = list(csv.reader(f))
    return rows[skip:]

def show_table(rows):
    html = "<table><tr>" + "".join(f"<th>{h}</th>" for h in rows[0]) + "</tr>"
    for r in rows[1:]:
        html += "<tr>" + "".join(f"<td>{c}</td>" for c in r) + "</tr>"
    display(HTML(html + "</table>"))

## ① 商品情報CSVの作成（build_plans_csv.py）

`sample/` のダミー同梱物マスタ・優先順位・緩衝材ルールから商品情報CSVを機械的に生成する。

In [ ]:
plans_csv = os.path.join(tempfile.mkdtemp(), "商品情報.csv")
print(run([PLANS_SCRIPT,
           "--items", os.path.join(SAMPLE, "同梱物マスタ.csv"),
           "--priority", os.path.join(SAMPLE, "優先順位.txt"),
           "--cushion-rules", os.path.join(SAMPLE, "緩衝材ルール.json"),
           "--out", plans_csv]))
show_table(read_csv_rows(plans_csv))

In [ ]:
# 期待値チェック：Aプランのみアクリルキーホルダー入り→緩衝材指定=1・BUBBLE_PACK、B/Cは空
rows = {r[0]: r for r in read_csv_rows(plans_csv, skip=1)}
assert rows["LT000255643004776689"][5:7] == ["1", "BUBBLE_PACK"], "Aプランの緩衝材判定が想定と異なる"
assert rows["LT000255643004776690"][5:7] == ["", ""], "Bプランの緩衝材判定が想定と異なる"
assert rows["LT000255643004776691"][5:7] == ["", ""], "Cプランの緩衝材判定が想定と異なる"
print("OK: ①の緩衝材判定は期待通り")

## ② 出荷依頼CSVの作成（build_opl_csv.py）

`mock/fixtures/` のダミー配送先回答・購入データ・突合リストと、①の商品情報CSVを突き合わせてオープンロジ出荷依頼CSVを生成する。

In [ ]:
opl_csv = os.path.join(tempfile.mkdtemp(), "出荷依頼.csv")
report = run([OPL_SCRIPT,
              "--bdash", os.path.join(FIXTURES, "配送先回答.csv"),
              "--orders", os.path.join(FIXTURES, "購入データ.csv"),
              "--konpo", os.path.join(FIXTURES, "突合リスト.csv"),
              "--plans", plans_csv,
              "--delivery", "4",
              "--out", opl_csv])
print(report)

all_rows = read_csv_rows(opl_csv)
header = all_rows[0]
cols = ["商品ID", "商品名", "出荷依頼数", "配送先氏名", "注文番号", "配送便指定", "緩衝材指定", "緩衝材", "品名"]
idx = [header.index(c) for c in cols]
show_table([cols] + [[r[i] for i in idx] for r in all_rows[2:]])

In [ ]:
# 検算：購入データの数量合計と出力の数量合計が一致すること（未発送が無い前提のfixtureなので差分ゼロのはず）
purchase_qty = len(read_csv_rows(os.path.join(FIXTURES, "購入データ.csv"), skip=1))
qty_idx = header.index("出荷依頼数")
output_qty = sum(int(r[qty_idx]) for r in all_rows[2:])
assert purchase_qty == output_qty, f"数量不一致: 購入{purchase_qty} vs 出力{output_qty}"
print(f"OK: 購入数量({purchase_qty})と出力数量({output_qty})が一致")

## ③ 緩衝材の同梱揃えルールの網羅検証

SKILL.mdのテスト履歴にある4パターン（Cのみ／B＋C／A＋C／A＋B＋C、優先度A→B→C）について、
同じ梱包内の全行が「優先度最上位プランの緩衝材・品名」に揃うことを確認する。
fixtureを使い回さず、その場で最小限の入力CSVを合成してbuild_opl_csv.pyに通す。

In [ ]:
def write_csv(path, header, rows):
    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)

AID = {"A": "LT000255643004776689", "B": "LT000255643004776690", "C": "LT000255643004776691"}
SCENARIOS = {
    "Cのみ": (["C"], "C"),
    "B+C": (["B", "C"], "B"),
    "A+C": (["A", "C"], "A"),
    "A+B+C": (["A", "B", "C"], "A"),
}

results = []
for name, (plans_in_pkg, expected_head) in SCENARIOS.items():
    d = tempfile.mkdtemp()
    member = "M90000001"
    orders_rows = [[AID[p], i + 1, f"TEST-{name}-{i}", "PDC", member] for i, p in enumerate(plans_in_pkg)]
    write_csv(os.path.join(d, "orders.csv"), ["AID", "No", "注文番号", "PDC", "RS会員ID"], orders_rows)
    write_csv(os.path.join(d, "konpo.csv"), ["RS会員ID", "梱包No"], [[member, f"TEST_{name}"]])
    write_csv(os.path.join(d, "bdash.csv"),
              ["回答ID", "顧客ID", "注文番号", "郵便番号", "都道府県", "住所", "建物名", "姓", "名", "電話番号", "メールアドレス"],
              [[1, "", f"TEST-{name}-0", "1000001", "東京都", "千代田区1-1-1", "", "テスト", "太郎", "09000000000", "t@example.com"]])
    out = os.path.join(d, "out.csv")
    run([OPL_SCRIPT, "--bdash", os.path.join(d, "bdash.csv"), "--orders", os.path.join(d, "orders.csv"),
         "--konpo", os.path.join(d, "konpo.csv"), "--plans", plans_csv, "--out", out])
    out_rows = read_csv_rows(out, skip=2)
    cushion_flags = {r[header.index("緩衝材指定")] for r in out_rows}
    cushion_mats = {r[header.index("緩衝材")] for r in out_rows}
    expected = rows[AID[expected_head]]
    ok = (cushion_flags == {expected[5]} and cushion_mats == {expected[6]})
    results.append((name, plans_in_pkg, expected_head, sorted(cushion_flags), sorted(cushion_mats), ok))

show_table([["構成", "同梱プラン", "期待する基準プラン", "緩衝材指定", "緩衝材", "判定"]] +
           [[n, "+".join(p), e, str(f), str(m), ("OK" if ok else "NG")] for n, p, e, f, m, ok in results])
assert all(ok for *_, ok in results), "緩衝材の同梱揃えルールに想定外の挙動あり"
print("OK: 4パターンすべてSKILL.mdのテスト履歴通り")